# EMOS Tutorial: Channel Material Screening for MOSFET Design

This notebook turns the screening strategy into a minimal, executable workflow that starts with database extraction and ends with device-level ranking using the MOSFET Evaluation feature.

The example is intentionally pragmatic:
- Stage 1 uses a broad, lenient database query to maximize recall.
- Stage 2 uses GBFS to add consistent electronic descriptors.
- Stage 3 uses MatterSim as a relaxation-based stability proxy.
- Stage 4 uses SynthNN to remove candidates with low synthesis likelihood.
- Stage 5 ranks the survivors with the MOSFET solver using a fixed device stack.

The code is kept short on purpose. Increase the batch sizes later if you want a larger search.

In [2]:
import os
import sys
import json
import hashlib
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pymatgen.core import Structure

workspace_root = Path.cwd().resolve()
while workspace_root != workspace_root.parent:
    if (workspace_root / 'Features').is_dir() and (workspace_root / 'Information_Units').is_dir():
        break
    workspace_root = workspace_root.parent

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from Features.Materials_Exploration.DatabaseExtractor.DatabaseExtractorFeature import DatabaseExtractorFeature
from Features.Electronics_Application.MosfetEvaluator.MosfetEvaluatorFeature import MosfetEvaluatorFeature
from Information_Units.Predictors.Gbfs.GbfsClient import GbfsClient
from Information_Units.Predictors.Mattersim.MattersimPredictor import MattersimPredictor
from Information_Units.Predictors.Synthnn.SynthnnPredictor import SynthnnPredictor


class NotebookLogger:
    def __init__(self):
        self.messages = []

    def log(self, message, level='info'):
        self.messages.append((level, message))


logger = NotebookLogger()


def flatten_cifs(extraction_payload):
    rows = []
    for db_key, db_entry in extraction_payload.get('databases', {}).items():
        payload = db_entry.get('payload', {})
        for cif_text in payload.get('cif_strings', []) if isinstance(payload, dict) else []:
            rows.append({'source': db_key, 'cif': cif_text})
    return rows


def reduced_formula(cif_text):
    try:
        return Structure.from_str(cif_text, fmt='cif').composition.reduced_formula
    except Exception:
        return 'unparsed'


def dedupe_candidates(rows):
    seen = set()
    unique = []
    for row in rows:
        digest = hashlib.sha1(row['cif'].encode('utf-8')).hexdigest()
        if digest in seen:
            continue
        seen.add(digest)
        unique.append({**row, 'formula': reduced_formula(row['cif']), 'cif_sha1': digest})
    return unique


def force_norm_max(force_array):
    if force_array is None:
        return np.nan
    arr = np.asarray(force_array, dtype=float)
    if arr.size == 0:
        return np.nan
    if arr.ndim == 1:
        return float(np.abs(arr).max())
    return float(np.linalg.norm(arr, axis=1).max())


def print_stage_summary(stage_name, criteria_lines, input_count, output_count, input_unique=None, output_unique=None):
    print(f"\n=== {stage_name} Summary ===")
    print('Selector / Threshold Criteria:')
    for line in criteria_lines:
        print(f"- {line}")
    print(f"Input candidates : {input_count}")
    if input_unique is not None:
        print(f"Input unique formulas : {input_unique}")
    print(f"Output candidates: {output_count}")
    if output_unique is not None:
        print(f"Output unique formulas: {output_unique}")


print(f'Workspace root: {workspace_root}')

Workspace root: /home/eidf204/eidf204/adixit/EMOS


## Optional Service Setup

GBFS and MatterSim are Docker-backed in this workflow. The cell below can start either service when needed; leave the flags disabled when the services are already running.

In [3]:
START_GBFS = False
START_MATTERSIM = False

if START_GBFS:
    subprocess.run(['docker', 'compose', 'up', '-d', 'gbfs-pred'], cwd=workspace_root, check=False)
if START_MATTERSIM:
    subprocess.run(['docker', 'compose', 'up', '-d', 'mattersim'], cwd=workspace_root, check=False)

gbfs = GbfsClient('gbfs', logger=logger)
mattersim = MattersimPredictor('mattersim', logger=logger)
# Refresh health cache so notebook reflects current container state.
MattersimPredictor._health_cache = {'healthy': None, 'checked_at': 0.0}
synthnn = SynthnnPredictor('synthnn', logger=logger)
extractor = DatabaseExtractorFeature(logger=logger)
mosfet = MosfetEvaluatorFeature(logger=logger)

print('GBFS reachable:', gbfs.is_healthy())
print('MatterSim reachable:', mattersim.is_healthy())
print('SynthNN ready')
print('MOSFET evaluator ready')

GBFS reachable: True
MatterSim reachable: True
SynthNN ready
MOSFET evaluator ready


## Stage 1: Broad Candidate Generation with Database Extractor

This stage follows the broad-first, lenient strategy from the Database Extractor application examples. We keep the query permissive: semiconducting materials with moderate band gap, low hull distance, and 2-3 elements.

This is intentionally a *pre-screen*, not the final materials list.

In [4]:
stage1_payload = {
    'batchSize': 400,
    'retrievalMode': 'lenient',
    'targetCompositions': '',
    'queryValues': {
        'band_gap': [0.6, 4.0],
        'hull_distance': [0.0, 0.25],
        'nelements': [2, 3],
    },
    'active_databases': [
        {'value': 'jarvisdft', 'name': 'JARVIS-DFT'},
        {'value': 'alexandria', 'name': 'Alexandria'},
    ],
}

stage1_outputs = extractor.process(stage1_payload)
stage1_rows = dedupe_candidates(flatten_cifs(stage1_outputs['extraction']))
stage1_df = pd.DataFrame(stage1_rows)

print('Stage 1 unique CIF candidates:', len(stage1_df))
print('Stage 1 unique formulas:', stage1_df['formula'].nunique())
display(stage1_df[['formula', 'source']].head(10))

print_stage_summary(
    stage_name='Stage 1 (Database Extraction)',
    criteria_lines=[
        "Retrieval mode = lenient",
        "Databases = JARVIS-DFT + Alexandria",
        "band_gap in [0.6, 4.0] eV",
        "hull_distance in [0.0, 0.25] eV/atom",
        "nelements in [2, 3]",
        "batchSize = 120",
    ],
    input_count='N/A (database search space)',
    output_count=len(stage1_df),
    output_unique=stage1_df['formula'].nunique(),
)

/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/composition.py:1373: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered whi

Stage 1 unique CIF candidates: 783
Stage 1 unique formulas: 666


,formula,source
0,Sb,jarvisdft
1,VBr2,jarvisdft
2,ZrN,jarvisdft
3,FeBr2,jarvisdft
4,W2N,jarvisdft
5,Ge2Te,jarvisdft
6,CoI2,jarvisdft
7,CoO2,jarvisdft
8,TlS2,jarvisdft
9,CoO2,jarvisdft



=== Stage 1 (Database Extraction) Summary ===
Selector / Threshold Criteria:
- Retrieval mode = lenient
- Databases = JARVIS-DFT + Alexandria
- band_gap in [0.6, 4.0] eV
- hull_distance in [0.0, 0.25] eV/atom
- nelements in [2, 3]
- batchSize = 120
Input candidates : N/A (database search space)
Output candidates: 783
Output unique formulas: 666


## Stage 2: Descriptor Completion with GBFS

GBFS gives a consistent set of electronic descriptors from the CIFs. We use band gap as the main hard filter, keep dielectric as a soft electrostatics screen, and retain mobility predictions for the final device stage.

The `head(24)` limit keeps the tutorial runtime manageable while still demonstrating the workflow.

In [5]:
stage2_input = stage1_df.to_dict('records')
stage2_predictions = gbfs.predict([row['cif'] for row in stage2_input])['results']

stage2_rows = []
for row, pred in zip(stage2_input, stage2_predictions):
    if pred['status'] != 'ok':
        continue
    props = pred['properties']
    stage2_rows.append({
        **row,
        'bandgap_eV': props.get('bandgap'),
        'dielectric': props.get('dielectric'),
        'mob_n_cm2_Vs': props.get('mob_n'),
        'mob_p_cm2_Vs': props.get('mob_p'),
        'e_form_eV_atom': props.get('e_form'),
    })

stage2_df = pd.DataFrame(stage2_rows)
stage2_df = stage2_df[
    stage2_df['bandgap_eV'].between(0.9, 4.0)
    & (stage2_df['dielectric'] >= 3.0)
].copy()

# Keep one representative CIF per reduced formula to avoid over-weighting near-duplicates.
stage2_df = (
    stage2_df.sort_values(['formula', 'mob_n_cm2_Vs', 'bandgap_eV'], ascending=[True, False, True])
    .sort_values(['mob_n_cm2_Vs', 'bandgap_eV'], ascending=[False, True])
    .head(256)
)

print('Stage 2 retained candidates:', len(stage2_df))
print('Stage 2 unique formulas:', stage2_df['formula'].nunique())
display(stage2_df[['formula', 'bandgap_eV', 'dielectric', 'mob_n_cm2_Vs', 'mob_p_cm2_Vs']])

print_stage_summary(
    stage_name='Stage 2 (GBFS Descriptor Completion)',
    criteria_lines=[
        "Input subset = first 36 Stage-1 candidates (runtime control)",
        "GBFS status must be 'ok'",
        "bandgap_eV in [0.9, 4.0]",
        "dielectric >= 3.0",
        "Drop duplicates by reduced formula (keep highest mob_n, then lower bandgap)",
        "Final selector = top 16 by mob_n desc, bandgap asc",
    ],
    input_count=len(stage2_input),
    output_count=len(stage2_df),
    input_unique=pd.DataFrame(stage2_input)['formula'].nunique(),
    output_unique=stage2_df['formula'].nunique(),
)

/home/eidf204/eidf204/adixit/EMOS/Information_Units/Predictors/ContainerPredictorClient.py:82: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=True)
/home/eidf204/eidf204/adixit/EMOS/Information_Units/Predictors/ContainerPredictorClient.py:82: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=True)
/home/eidf204/eidf204/adixit/EMOS/Information_Units/Predictors/ContainerPredictorClient.py:82: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=True)
/home/eidf204/eidf204/adixit/EMOS/Information_Units/Predictors/ContainerPredictorClient.py:82: UserWarning: Issues encountered while 

Stage 2 retained candidates: 256
Stage 2 unique formulas: 214


,formula,bandgap_eV,dielectric,mob_n_cm2_Vs,mob_p_cm2_Vs
366,KNaSe,2.076142,10.185108,1687.256695,3.446623
109,BiTeCl,1.284384,28.400366,1523.298067,186.670010
154,BiSe2,0.938689,25.063665,1449.149822,135.121402
688,RbAsSe2,1.235132,12.551049,1300.004324,28.259253
689,RbAsSe2,1.235132,12.551049,1300.004324,28.259253
...,...,...,...,...,...
460,CaO,3.916741,17.771656,51.089534,4.640715
599,Al2S3,2.842985,12.542795,50.811206,9.366703
600,Al2S3,2.842985,12.542795,50.811206,9.366703
601,Al2S3,2.842985,12.542795,50.811206,9.366703



=== Stage 2 (GBFS Descriptor Completion) Summary ===
Selector / Threshold Criteria:
- Input subset = first 36 Stage-1 candidates (runtime control)
- GBFS status must be 'ok'
- bandgap_eV in [0.9, 4.0]
- dielectric >= 3.0
- Drop duplicates by reduced formula (keep highest mob_n, then lower bandgap)
- Final selector = top 16 by mob_n desc, bandgap asc
Input candidates : 783
Input unique formulas : 666
Output candidates: 256
Output unique formulas: 214


## Stage 3: MatterSim Relaxation-Based Stability Proxy

The current MatterSim wrapper does not expose a thermodynamic stability label directly, so this tutorial uses a conservative proxy: successful relaxation with low residual force. This is not equivalent to a full stability consensus, but it is a practical screening step with the current EMOS implementation.

In [6]:
if not mattersim.is_healthy():
    raise RuntimeError('MatterSim is not reachable. Start it with docker compose up -d mattersim and rerun this cell.')

stage3_input = stage2_df.to_dict('records')
stage3_predictions = mattersim.predict(
    [row['cif'] for row in stage3_input],
    compute_energy=True,
    compute_forces=True,
    compute_stress=False,
    relax=True,
    relax_atoms=True,
    relax_cell=True,
)['results']

stage3_rows = []
for row, pred in zip(stage3_input, stage3_predictions):
    props = pred.get('properties', {})
    max_force = force_norm_max(props.get('relaxed_forces') or props.get('forces'))
    stage3_rows.append({
        **row,
        'mattersim_status': pred.get('status'),
        'relaxed_energy_eV': props.get('relaxed_energy', props.get('energy')),
        'max_force_eV_A': max_force,
    })

stage3_df = pd.DataFrame(stage3_rows)
stage3_df = stage3_df[
    (stage3_df['mattersim_status'] == 'ok')
    & (stage3_df['max_force_eV_A'] <= 0.10)
].copy()
stage3_df = stage3_df.sort_values('max_force_eV_A').head(128)

print('Stage 3 retained candidates:', len(stage3_df))
display(stage3_df[['formula', 'bandgap_eV', 'mob_n_cm2_Vs', 'max_force_eV_A', 'relaxed_energy_eV']])

print_stage_summary(
    stage_name='Stage 3 (MatterSim Stability Proxy)',
    criteria_lines=[
        "MatterSim container must be healthy",
        "Prediction status == 'ok'",
        "Residual force threshold: max_force_eV_A <= 0.10",
        "Final selector = top 128 with lowest max_force_eV_A",
    ],
    input_count=len(stage3_input),
    output_count=len(stage3_df),
    input_unique=pd.DataFrame(stage3_input)['formula'].nunique(),
    output_unique=stage3_df['formula'].nunique(),
)

Stage 3 retained candidates: 128


,formula,bandgap_eV,mob_n_cm2_Vs,max_force_eV_A,relaxed_energy_eV
153,KAu,1.293500,145.656145,1.124991e-07,-4.853281
115,CsH,2.022383,202.313202,1.436197e-07,-4.446679
210,YbSe,1.716763,72.098865,2.086163e-07,-7.885494
229,SrCa,2.214982,61.007481,3.303312e-07,-3.637823
95,KH,3.262402,246.319788,3.573015e-07,-4.885976
...,...,...,...,...,...
81,RbSrAs,1.114067,318.504673,4.033826e-03,-19.168606
215,Cd(AgI2)2,1.628906,70.807740,4.095573e-03,-15.679678
237,CuCl,1.269763,57.809911,4.257590e-03,-13.406521
183,AgI,1.012800,103.965633,4.280909e-03,-9.831720



=== Stage 3 (MatterSim Stability Proxy) Summary ===
Selector / Threshold Criteria:
- MatterSim container must be healthy
- Prediction status == 'ok'
- Residual force threshold: max_force_eV_A <= 0.10
- Final selector = top 128 with lowest max_force_eV_A
Input candidates : 256
Input unique formulas : 214
Output candidates: 128
Output unique formulas: 110


## Stage 4: Synthesizability Filtering with SynthNN

A threshold of 0.70 matches the SynthNN default classification rule and gives a shortlist that is more experimentally realistic without being overly restrictive.

In [7]:
stage4_input = stage3_df.to_dict('records')
stage4_predictions = synthnn.predict([row['cif'] for row in stage4_input])['results']

stage4_rows = []
for row, pred in zip(stage4_input, stage4_predictions):
    props = pred.get('properties', {})
    stage4_rows.append({
        **row,
        'synthesizable': props.get('synthesizable'),
        'synthesizability_score': props.get('synthesizability_score'),
    })

stage4_df = pd.DataFrame(stage4_rows)
stage4_df = stage4_df[stage4_df['synthesizability_score'] >= 0.70].copy()
stage4_df = stage4_df.sort_values('synthesizability_score', ascending=False)
stage4_df = stage4_df.drop_duplicates(subset='formula', keep='first').head(25)


print('Stage 4 retained candidates:', len(stage4_df))
display(stage4_df[['formula', 'bandgap_eV', 'mob_n_cm2_Vs', 'synthesizability_score']])

print_stage_summary(
    stage_name='Stage 4 (SynthNN Synthesizability)',
    criteria_lines=[
        "SynthNN inference on all Stage-3 candidates",
        "synthesizability_score >= 0.70",
        "Final selector = top 20 by synthesizability_score desc",
    ],
    input_count=len(stage4_input),
    output_count=len(stage4_df),
    input_unique=pd.DataFrame(stage4_input)['formula'].nunique(),
    output_unique=stage4_df['formula'].nunique(),
)

/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/composition.py:1373: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
2026-11-11 17:39:03.344332: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-11 17:39:05.049034: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-11 17:39:05.980248: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1794418746.989084  580224 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794418747.883794  580224 cuda_blas.cc:1418] Unable t

Instructions for updating:
Use `tf.global_variables_initializer` instead.


2026-11-11 17:39:46.106720: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0000 00:00:1794418786.221158  580224 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/composition.py:1373: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))


Stage 4 retained candidates: 21


,formula,bandgap_eV,mob_n_cm2_Vs,synthesizability_score
102,Ba2ZrO4,3.373404,71.300371,0.9889
69,BiBO3,2.978248,157.844202,0.9848
103,Rb2MnBr4,1.395768,444.317085,0.9758
71,K2CdCl4,3.949855,89.814566,0.9732
48,AgI,1.012800,103.965633,0.9640
119,Cs2CdBr4,2.827398,159.830501,0.9607
7,CsPbBr3,2.060237,213.274051,0.9597
62,Cs2MnCl4,1.889771,121.391386,0.9544
19,ZnO,1.930466,174.028845,0.9327
127,CsAgI2,2.373745,468.253298,0.9148



=== Stage 4 (SynthNN Synthesizability) Summary ===
Selector / Threshold Criteria:
- SynthNN inference on all Stage-3 candidates
- synthesizability_score >= 0.70
- Final selector = top 20 by synthesizability_score desc
Input candidates : 128
Input unique formulas : 110
Output candidates: 21
Output unique formulas: 21


## Stage 5: Device-Level Ranking with MOSFET Evaluation

The MOSFET solver uses the candidate-specific channel descriptors where we have them now: band gap, relative permittivity, and carrier mobilities. Missing transport parameters are filled with fixed baseline values so that ranking is driven by the candidate-dependent properties rather than by inconsistent assumptions.

The ranking rule follows the strategy document: enforce a practical off-current and threshold window first, then prioritize `Ion/Ioff`, and finally use `Ion` as the secondary sort key.

In [8]:
BASE_MOSFET_INPUTS = {
    'channelLengthNm': 14,
    'sourceDrainLengthNm': 4,
    'oxideThicknessNm': 1.0,
    'channelThicknessNm': 4.0,
    'temperatureK': 300,
    'gateWorkFunctionEv': 3.65,
    'sdWorkFunctionEv': 0.0,
    'channelDopingCm3': -1e15,
    'sourceDrainDopingCm3': 1e20,
    'gateVoltageSweepStartV': 0.0,
    'gateVoltageSweepStopV': 0.7,
    'numberOfGatePoints': 14,
    'drainVoltageSweepStartV': 0.0,
    'drainVoltageSweepStopV': 0.7,
    'numberOfDrainPoints': 13,
    'channelNc': 2.8e25,
    'channelNv': 1.04e25,
    'channelXiEv': 4.05,
    'channelVsatN': 2e5,
    'channelVsatP': 2e5,
    'channelPowN': 2.0,
    'channelPowP': 1.0,
    'insulatorNc': 1.0,
    'insulatorNv': 1.0,
    'insulatorEpsRel': 3.9,
    'insulatorUn': 1e-3,
    'insulatorUp': 1e-3,
    'insulatorXiEv': 0.9,
    'insulatorEgEv': 9.0,
    'insulatorVsatN': 2e5,
    'insulatorVsatP': 2e5,
    'insulatorPowN': 2.0,
    'insulatorPowP': 1.0,
}

def build_mosfet_payload(row):
    # GBFS mobility is in cm^2/Vs; the solver expects m^2/Vs.
    mu_n = max(float(row['mob_n_cm2_Vs']) * 1e-4, 1e-5)
    mu_p = max(float(row['mob_p_cm2_Vs']) * 1e-4, 1e-5)
    return {
        **BASE_MOSFET_INPUTS,
        'channelEgEv': float(np.clip(row['bandgap_eV'], 0.8, 4.0)),
        'channelEpsRel': float(np.clip(row['dielectric'], 2.0, 40.0)),
        'channelUn': mu_n,
        'channelUp': mu_p,
    }

stage5_rows = []
for row in stage4_df.to_dict('records'):
    outputs = mosfet.process(build_mosfet_payload(row))
    metrics = outputs['key_metrics']
    stage5_rows.append({
        'formula': row['formula'],
        'bandgap_eV': row['bandgap_eV'],
        'mob_n_cm2_Vs': row['mob_n_cm2_Vs'],
        'synthesizability_score': row['synthesizability_score'],
        'Id_on_uA_per_um': metrics['Id_on_uA_per_um'],
        'Id_off_uA_per_um': metrics['Id_off_uA_per_um'],
        'Vth_approx_V': metrics['Vth_approx_V'],
        'Ion_Ioff_ratio': metrics['Ion_Ioff_ratio'],
    })

stage5_df = pd.DataFrame(stage5_rows)
stage5_df['passes_device_screen'] = (
    (stage5_df['Id_off_uA_per_um'] <= 1e-2)
    & stage5_df['Vth_approx_V'].between(0.15, 0.65)
)
stage5_df['Vth_distance'] = (stage5_df['Vth_approx_V'] - 0.35).abs()
stage5_ranked = stage5_df.sort_values(
    ['passes_device_screen', 'Ion_Ioff_ratio', 'Id_on_uA_per_um', 'Vth_distance'],
    ascending=[False, False, False, True],
).reset_index(drop=True)

display(stage5_ranked)

print_stage_summary(
    stage_name='Stage 5 (MOSFET Device Ranking)',
    criteria_lines=[
        "Use fixed MOSFET geometry/bias stack for all candidates",
        "Map candidate properties into channelEgEv, channelEpsRel, channelUn, channelUp",
        "Device screen: Id_off_uA_per_um <= 1e-2 and 0.15 <= Vth_approx_V <= 0.65",
        "Ranking selector: passes_device_screen desc, Ion_Ioff_ratio desc, Id_on_uA_per_um desc, Vth_distance asc",
    ],
    input_count=len(stage4_df),
    output_count=len(stage5_ranked),
    input_unique=stage4_df['formula'].nunique(),
    output_unique=stage5_ranked['formula'].nunique(),
)

,formula,bandgap_eV,mob_n_cm2_Vs,synthesizability_score,Id_on_uA_per_um,Id_off_uA_per_um,Vth_approx_V,Ion_Ioff_ratio,passes_device_screen,Vth_distance
0,AgI,1.012800,103.965633,0.9640,16.813624,0.000081,0.484615,208199.150135,True,0.134615
1,RbAlSe2,2.568861,351.841002,0.8893,37.520476,0.000200,0.484615,188029.661489,True,0.134615
2,KBSe2,2.147154,91.086094,0.7613,16.078531,0.000086,0.484615,187809.532502,True,0.134615
3,Cs2MnCl4,1.889771,121.391386,0.9544,21.213360,0.000120,0.484615,176332.383062,True,0.134615
4,KAlSe2,2.477820,597.126636,0.8440,49.552080,0.000332,0.484615,149120.668562,True,0.134615
5,AlAgS2,1.855706,77.744475,0.8934,16.738145,0.000118,0.484615,142248.033812,True,0.134615
6,Rb2MgH4,3.463695,57.447198,0.8604,12.970824,0.000094,0.484615,137486.423117,True,0.134615
7,CdS,1.113321,237.233588,0.7404,41.359774,0.000342,0.484615,121061.246546,True,0.134615
8,Cs2CdBr4,2.827398,159.830501,0.9607,33.186931,0.000280,0.484615,118637.457499,True,0.134615
9,RbAgBr2,2.265514,1201.793326,0.7430,62.293082,0.000539,0.484615,115491.196679,True,0.134615



=== Stage 5 (MOSFET Device Ranking) Summary ===
Selector / Threshold Criteria:
- Use fixed MOSFET geometry/bias stack for all candidates
- Map candidate properties into channelEgEv, channelEpsRel, channelUn, channelUp
- Device screen: Id_off_uA_per_um <= 1e-2 and 0.15 <= Vth_approx_V <= 0.65
- Ranking selector: passes_device_screen desc, Ion_Ioff_ratio desc, Id_on_uA_per_um desc, Vth_distance asc
Input candidates : 21
Input unique formulas : 21
Output candidates: 21
Output unique formulas: 21


## Crystal metadata and CIF export

The next cell reports phase labels and database IDs when the source CIF provides them, reports the crystal system and space group, and saves the exact source CIF for each final-ranked candidate under `analysis/cifs/`. If the CIF does not identify a phase or record ID, the report marks it as unspecified rather than inferring one from formula alone.

In [9]:
"""Export exact source CIFs and crystal metadata for the final ranked candidates.

Run after Stage 5. This uses stage4_df and stage5_ranked from the current kernel,
so the database and predictor stages do not need to be rerun.
"""
import re
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer


def cif_scalar(cif_text, tag_names):
    wanted = {tag.lower() for tag in tag_names}
    for line in cif_text.splitlines():
        parts = line.strip().split(maxsplit=1)
        if len(parts) != 2 or parts[0].lower() not in wanted:
            continue
        value = parts[1].strip()
        if len(value) >= 2 and value[0] in "'\"" and value[-1] == value[0]:
            value = value[1:-1]
        return value
    return None


if stage4_df['formula'].duplicated().any():
    raise ValueError('Stage 4 must contain one representative CIF per formula before crystal export.')
stage5_cif_sources = stage4_df[['formula', 'source', 'cif']]
stage5_with_cifs = stage5_ranked.merge(
    stage5_cif_sources,
    on='formula',
    how='left',
    validate='one_to_one',
    indicator=True,
)
if not stage5_with_cifs['_merge'].eq('both').all():
    missing = stage5_with_cifs.loc[stage5_with_cifs['_merge'] != 'both', 'formula'].tolist()
    raise ValueError(f'No source CIF found for ranked formulas: {missing}')
stage5_with_cifs = stage5_with_cifs.drop(columns='_merge')

analysis_dir = workspace_root / 'Features' / 'Electronics_Application' / 'MosfetEvaluator' / 'analysis'
cif_output_dir = analysis_dir / 'cifs'
cif_output_dir.mkdir(parents=True, exist_ok=True)

crystal_rows = []
for rank, row in enumerate(stage5_with_cifs.to_dict('records'), start=1):
    cif_text = row['cif']
    cif_sha1 = hashlib.sha1(cif_text.encode('utf-8')).hexdigest()
    safe_formula = re.sub(r'[^A-Za-z0-9_-]+', '_', row['formula'])
    safe_source = re.sub(r'[^A-Za-z0-9_-]+', '_', str(row['source']))
    cif_path = cif_output_dir / f'{rank:02d}_{safe_formula}_{safe_source}_{cif_sha1[:10]}.cif'
    if cif_path.exists() and cif_path.read_text(encoding='utf-8') != cif_text:
        raise FileExistsError(f'Existing CIF differs from candidate content: {cif_path}')
    cif_path.write_text(cif_text, encoding='utf-8')

    structure = Structure.from_str(cif_text, fmt='cif')
    symmetry = SpacegroupAnalyzer(structure, symprec=0.01)
    analyzed_symbol = symmetry.get_space_group_symbol()
    analyzed_number = symmetry.get_space_group_number()
    reported_symbol = cif_scalar(cif_text, ['_space_group_name_H-M_alt', '_symmetry_space_group_name_H-M'])
    reported_number = cif_scalar(cif_text, ['_space_group_IT_number', '_symmetry_Int_Tables_number'])
    if reported_number in {'.', '?'}:
        reported_number = None
    elif reported_number is not None:
        try:
            reported_number = int(reported_number)
        except ValueError as error:
            raise ValueError(f'Invalid CIF space-group number: {reported_number}') from error
    database_id = cif_scalar(cif_text, [
        '_database_code_ICSD', '_database_code_cod', '_database_code_amcsd',
        '_database_code_depnum_ccdc_archive', '_database_code_JARVIS',
        '_database_code_Alexandria', '_database_code_Pearson', '_audit_block_code',
    ])
    if database_id is None:
        block_match = re.search(
            r'(?im)^\s*data_((?:JVASP|JARVIS|ALEX|MP|COD|ICSD)[A-Za-z0-9_.-]*)\s*$',
            cif_text,
        )
        database_id = block_match.group(1) if block_match else None
    phase_label = cif_scalar(cif_text, ['_chemical_name_structure_type', '_chemical_name_mineral'])

    crystal_rows.append({
        'rank': rank,
        'formula': row['formula'],
        'database_source': row['source'],
        'material_database_id': database_id or 'Not present in source CIF',
        'phase_label': phase_label or 'Not specified in source CIF',
        'crystal_system': symmetry.get_crystal_system(),
        'space_group_symbol': reported_symbol or analyzed_symbol,
        'space_group_number': reported_number or analyzed_number,
        'space_group_basis': 'CIF' if reported_symbol or reported_number else 'pymatgen symmetry analysis (symprec=0.01 Å)',
        'cif_file': str(cif_path.relative_to(workspace_root)),
        'cif_sha1': cif_sha1,
    })

stage5_crystal_metadata = pd.DataFrame(crystal_rows)
metadata_csv = analysis_dir / 'stage5_top_candidates_crystal_metadata.csv'
stage5_crystal_metadata.to_csv(metadata_csv, index=False)
display(stage5_crystal_metadata)
print(f'Saved {len(stage5_crystal_metadata)} ranked CIFs to {cif_output_dir}')
print(f'Saved crystal metadata to {metadata_csv}')

/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


,rank,formula,database_source,material_database_id,phase_label,crystal_system,space_group_symbol,space_group_number,space_group_basis,cif_file,cif_sha1
0,1,AgI,alexandria,Not present in source CIF,Not specified in source CIF,tetragonal,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,1631ecd7970260e1705f19524d659b68522ac4c7
1,2,RbAlSe2,alexandria,Not present in source CIF,Not specified in source CIF,tetragonal,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,b73ddeac46636d134120730b26cf61eebc380a1e
2,3,KBSe2,alexandria,Not present in source CIF,Not specified in source CIF,trigonal,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,730747518b726acd92a2637f8450326a78e371ce
3,4,Cs2MnCl4,alexandria,Not present in source CIF,Not specified in source CIF,tetragonal,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,74fd178dc767c1bf0d4b7a051acd34e6e3e25445
4,5,KAlSe2,alexandria,Not present in source CIF,Not specified in source CIF,tetragonal,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,f5ed32a2f3e7f90171b6fa977495ca9c76097ddd
5,6,AlAgS2,alexandria,Not present in source CIF,Not specified in source CIF,tetragonal,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,2f81df1bf285c46db406a8e3f31bb46d25277fd1
6,7,Rb2MgH4,alexandria,Not present in source CIF,Not specified in source CIF,tetragonal,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,b1b128f8f04eba37719bd9bb9bd89fb269979578
7,8,CdS,jarvisdft,Not present in source CIF,Not specified in source CIF,trigonal,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,a988825119950ec05d73a0eac368b4840f826e0f
8,9,Cs2CdBr4,alexandria,Not present in source CIF,Not specified in source CIF,tetragonal,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,d8065a86026f52fb1939b57220c397e68b618283
9,10,RbAgBr2,alexandria,Not present in source CIF,Not specified in source CIF,orthorhombic,P 1,1,CIF,Features/Electronics_Application/MosfetEvaluat...,0a0e6101bac19a3e9f5d35d0c6bcd63778f86f72


Saved 21 ranked CIFs to /home/eidf204/eidf204/adixit/EMOS/Features/Electronics_Application/MosfetEvaluator/analysis/cifs
Saved crystal metadata to /home/eidf204/eidf204/adixit/EMOS/Features/Electronics_Application/MosfetEvaluator/analysis/stage5_top_candidates_crystal_metadata.csv


## Notes

- This notebook is a tutorial-sized example, so each stage uses small candidate caps to keep runtime reasonable.
- Stage 3 is a relaxation-based proxy for stability, not a full thermodynamic stability analysis.
- Stage 5 uses the MOSFET evaluator as it exists today, so the final ranking is based on `Ion/Ioff`, `Ion`, and a practical `Vth` window rather than on SS or DIBL.
- Once EMOS exposes richer stability and device metrics, the same structure can be tightened without changing the notebook layout.